# Healthcare Claims Fraud Detection

### Objective
Build a machine learning model to identify potentially fraudulent healthcare providers using beneficiary, inpatient, and outpatient claims data.

### Target Variable
**PotentialFraud**
- Yes = Fraudulent Provider
- No = Non-Fraudulent Provider

In [1]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
print("Libraries Imported Successfully")

Libraries Imported Successfully


## Load Datasets

In [2]:
train = pd.read_csv("../data/raw/Train.csv")
beneficiary = pd.read_csv("../data/raw/Train_Beneficiarydata.csv")
inpatient = pd.read_csv("../data/raw/Train_Inpatientdata.csv")
outpatient = pd.read_csv("../data/raw/Train_Outpatientdata.csv")
print("Datasets Loaded Successfully")

Datasets Loaded Successfully


## Dataset Dimensions

In [3]:
print("Fraud Labels:", train.shape)
print("Beneficiary:", beneficiary.shape)
print("Inpatient:", inpatient.shape)
print("Outpatient:", outpatient.shape)

Fraud Labels: (5410, 2)
Beneficiary: (138556, 25)
Inpatient: (40474, 30)
Outpatient: (517737, 27)


## Preview Fraud Labels Dataset

In [4]:
train.head()

,Provider,PotentialFraud
0,PRV51001,No
1,PRV51003,Yes
2,PRV51004,No
3,PRV51005,Yes
4,PRV51007,No


## Fraud Distribution

In [5]:
train["PotentialFraud"].value_counts()

PotentialFraud
No     4904
Yes     506
Name: count, dtype: int64

## Fraud Percentage

In [7]:
(
    train["PotentialFraud"].value_counts(normalize=True)
    .mul(100)
    .round(2)
)

PotentialFraud
No     90.65
Yes     9.35
Name: proportion, dtype: float64

### Observation

The dataset is highly imbalanced, with only 9.35% of providers labeled as potentially fraudulent.

This indicates that fraud detection is a rare-event classification problem. Therefore, model evaluation should focus on Precision, Recall, F1-Score, and ROC-AUC rather than Accuracy alone.

## Fraud Distribution Visualization

In [8]:
import plotly.express as px
fraud_counts = (
    train["PotentialFraud"]
    .value_counts()
    .reset_index()
)
fraud_counts.columns = [ "PotentialFraud", "Count" ]
fig = px.pie(
    fraud_counts,
    names="PotentialFraud",
    values="Count",
    hole=0.5,
    title="Fraud vs Non-Fraud Providers"
)
fig.update_traces(
    textposition="inside",
    textinfo="percent+label"
)
fig.update_layout(
    title_x=0.5
)
fig.show()

## Beneficiary Dataset Overview

In [9]:
beneficiary.head()

,BeneID,DOB,DOD,Gender,Race,RenalDiseaseIndicator,State,County,NoOfMonths_PartACov,NoOfMonths_PartBCov,ChronicCond_Alzheimer,ChronicCond_Heartfailure,ChronicCond_KidneyDisease,ChronicCond_Cancer,ChronicCond_ObstrPulmonary,ChronicCond_Depression,ChronicCond_Diabetes,ChronicCond_IschemicHeart,ChronicCond_Osteoporasis,ChronicCond_rheumatoidarthritis,ChronicCond_stroke,IPAnnualReimbursementAmt,IPAnnualDeductibleAmt,OPAnnualReimbursementAmt,OPAnnualDeductibleAmt
0,BENE11001,1943-01-01,NaN,1,1,0,39,230,12,12,1,2,1,2,2,1,1,1,2,1,1,36000,3204,60,70
1,BENE11002,1936-09-01,NaN,2,1,0,39,280,12,12,2,2,2,2,2,2,2,2,2,2,2,0,0,30,50
2,BENE11003,1936-08-01,NaN,1,1,0,52,590,12,12,1,2,2,2,2,2,2,1,2,2,2,0,0,90,40
3,BENE11004,1922-07-01,NaN,1,1,0,39,270,12,12,1,1,2,2,2,2,1,1,1,1,2,0,0,1810,760
4,BENE11005,1935-09-01,NaN,1,1,0,24,680,12,12,2,2,2,2,1,2,1,2,2,2,2,0,0,1790,1200


## Missing Value Analysis - Beneficiary Dataset

In [11]:
beneficiary.isnull().sum().sort_values(ascending=False).head(10)

DOD                                137135
BeneID                                  0
ChronicCond_Cancer                      0
OPAnnualReimbursementAmt                0
IPAnnualDeductibleAmt                   0
IPAnnualReimbursementAmt                0
ChronicCond_stroke                      0
ChronicCond_rheumatoidarthritis         0
ChronicCond_Osteoporasis                0
ChronicCond_IschemicHeart               0
dtype: int64

### Observation

The DOD (Date of Death) column contains approximately 99% missing values.

This is expected because most beneficiaries are alive. Therefore, these missing values do not indicate poor data quality and should not be removed.

A future feature such as `IsDeceased` can be derived from this column.

## Missing Value Percentage

In [12]:
missing_pct = (beneficiary.isnull().sum()
    / len(beneficiary) * 100
).sort_values(ascending=False)
missing_pct.head(10)

DOD                                98.974422
BeneID                              0.000000
ChronicCond_Cancer                  0.000000
OPAnnualReimbursementAmt            0.000000
IPAnnualDeductibleAmt               0.000000
IPAnnualReimbursementAmt            0.000000
ChronicCond_stroke                  0.000000
ChronicCond_rheumatoidarthritis     0.000000
ChronicCond_Osteoporasis            0.000000
ChronicCond_IschemicHeart           0.000000
dtype: float64

### Observation

The beneficiary dataset contains no significant missing values except for the DOD (Date of Death) column.

Since missing DOD values indicate that a beneficiary is alive, these records will be retained. Overall, the dataset exhibits excellent data quality and requires minimal missing value treatment.

## Beneficiary Data Types

In [13]:
beneficiary.info()

<class 'pandas.DataFrame'>
RangeIndex: 138556 entries, 0 to 138555
Data columns (total 25 columns):
 #   Column                           Non-Null Count   Dtype
---  ------                           --------------   -----
 0   BeneID                           138556 non-null  str  
 1   DOB                              138556 non-null  str  
 2   DOD                              1421 non-null    str  
 3   Gender                           138556 non-null  int64
 4   Race                             138556 non-null  int64
 5   RenalDiseaseIndicator            138556 non-null  str  
 6   State                            138556 non-null  int64
 7   County                           138556 non-null  int64
 8   NoOfMonths_PartACov              138556 non-null  int64
 9   NoOfMonths_PartBCov              138556 non-null  int64
 10  ChronicCond_Alzheimer            138556 non-null  int64
 11  ChronicCond_Heartfailure         138556 non-null  int64
 12  ChronicCond_KidneyDisease        138556 n

## Understanding Chronic Condition Encoding

In [15]:
beneficiary[
    ["ChronicCond_Alzheimer", "ChronicCond_Heartfailure", "ChronicCond_Diabetes"]
].nunique()

ChronicCond_Alzheimer       2
ChronicCond_Heartfailure    2
ChronicCond_Diabetes        2
dtype: int64

## Inspect Chronic Condition Values

In [16]:
print( "Alzheimer:", beneficiary["ChronicCond_Alzheimer"].unique())
print( "Heart Failure:", beneficiary["ChronicCond_Heartfailure"].unique())
print( "Diabetes:", beneficiary["ChronicCond_Diabetes"].unique())

Alzheimer: [1 2]
Heart Failure: [2 1]
Diabetes: [1 2]


## Verify Chronic Condition Encoding

In [17]:
for col in ["ChronicCond_Alzheimer", "ChronicCond_Heartfailure", "ChronicCond_Diabetes"]:
    print(f"\n{col}")
    print(beneficiary[col].value_counts())


ChronicCond_Alzheimer
ChronicCond_Alzheimer
2    92530
1    46026
Name: count, dtype: int64

ChronicCond_Heartfailure
ChronicCond_Heartfailure
2    70154
1    68402
Name: count, dtype: int64

ChronicCond_Diabetes
ChronicCond_Diabetes
1    83391
2    55165
Name: count, dtype: int64


## Convert Date Columns

In [18]:
beneficiary["DOB"] = pd.to_datetime(beneficiary["DOB"])
beneficiary["DOD"] = pd.to_datetime(beneficiary["DOD"])
print("Date conversion completed")

Date conversion completed


## Feature Engineering - Patient Age

In [19]:
beneficiary["Age"] = (pd.Timestamp.today().year - beneficiary["DOB"].dt.year)
beneficiary["Age"].describe()

count    138556.000000
mean         90.128663
std          12.724354
min          43.000000
25%          85.000000
50%          91.000000
75%          98.000000
max         117.000000
Name: Age, dtype: float64

### Observation

The beneficiary population is predominantly elderly, with a median age of 91 years.

This aligns with the Medicare population, which primarily consists of senior citizens. Age is expected to play an important role in healthcare utilization and reimbursement patterns.

## Feature Engineering - Total Chronic Conditions

In [20]:
chronic_cols = [
    "ChronicCond_Alzheimer",
    "ChronicCond_Heartfailure",
    "ChronicCond_KidneyDisease",
    "ChronicCond_Cancer",
    "ChronicCond_ObstrPulmonary",
    "ChronicCond_Depression",
    "ChronicCond_Diabetes",
    "ChronicCond_IschemicHeart",
    "ChronicCond_Osteoporasis",
    "ChronicCond_rheumatoidarthritis",
    "ChronicCond_stroke"
]
beneficiary["TotalChronicConditions"] = (beneficiary[chronic_cols].replace({2: 0}).sum(axis=1))
beneficiary["TotalChronicConditions"].describe()

count    138556.000000
mean          3.739131
std           2.346638
min           0.000000
25%           2.000000
50%           4.000000
75%           5.000000
max          11.000000
Name: TotalChronicConditions, dtype: float64

### Observation

Beneficiaries have an average of 3.74 chronic conditions, indicating a medically complex population.

This feature serves as a proxy for overall patient health burden and may help explain variations in healthcare utilization, reimbursements, and provider claim activity.

## Distribution of Total Chronic Conditions

In [21]:
chronic_dist = (
    beneficiary["TotalChronicConditions"]
    .value_counts()
    .sort_index()
    .reset_index()
)
chronic_dist.columns = ["TotalChronicConditions", "Count"]
fig = px.bar(
    chronic_dist,
    x="TotalChronicConditions",
    y="Count",
    text="Count",
    title="Distribution of Total Chronic Conditions"
)
fig.update_layout(title_x=0.5)
fig.show()

### Observation

The majority of beneficiaries have between 3 and 5 chronic conditions, with the distribution gradually declining for higher condition counts.

The feature appears well-formed and provides a useful summary of patient health complexity that can later be aggregated at the provider level.

## Feature Engineering - Deceased Beneficiary Indicator

In [22]:
beneficiary["IsDeceased"] = np.where(
    beneficiary["DOD"].notna(), 1, 0)
beneficiary["IsDeceased"].value_counts()

IsDeceased
0    137135
1      1421
Name: count, dtype: int64

### Observation

Approximately 1.03% of beneficiaries are deceased.

The vast majority of beneficiaries are alive, which is expected for Medicare claims data. The IsDeceased feature may provide useful signals when aggregated at the provider level.

# Inpatient Claims Analysis

Inpatient claims represent hospital admissions and generally involve higher costs than outpatient visits.

This dataset contains provider information, physicians, diagnoses, admission dates, discharge dates, and reimbursement amounts.

In [30]:
inpatient.head()

,BeneID,ClaimID,ClaimStartDt,ClaimEndDt,Provider,InscClaimAmtReimbursed,AttendingPhysician,OperatingPhysician,OtherPhysician,AdmissionDt,ClmAdmitDiagnosisCode,DeductibleAmtPaid,DischargeDt,DiagnosisGroupCode,ClmDiagnosisCode_1,ClmDiagnosisCode_2,ClmDiagnosisCode_3,ClmDiagnosisCode_4,ClmDiagnosisCode_5,ClmDiagnosisCode_6,ClmDiagnosisCode_7,ClmDiagnosisCode_8,ClmDiagnosisCode_9,ClmDiagnosisCode_10,ClmProcedureCode_1,ClmProcedureCode_2,ClmProcedureCode_3,ClmProcedureCode_4,ClmProcedureCode_5,ClmProcedureCode_6,LengthOfStay,DiagnosisCount
0,BENE11001,CLM46614,2009-04-12,2009-04-18,PRV55912,26000,PHY390922,NaN,NaN,2009-04-12,7866,1068.0,2009-04-18,201,1970,4019,5853,7843,2768,71590,2724,19889,5849,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,9
1,BENE11001,CLM66048,2009-08-31,2009-09-02,PRV55907,5000,PHY318495,PHY318495,NaN,2009-08-31,6186,1068.0,2009-09-02,750,6186,2948,56400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7092.0,NaN,NaN,NaN,NaN,NaN,2,3
2,BENE11001,CLM68358,2009-09-17,2009-09-20,PRV56046,5000,PHY372395,NaN,PHY324689,2009-09-17,29590,1068.0,2009-09-20,883,29623,30390,71690,34590,V1581,32723,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,6
3,BENE11011,CLM38412,2009-02-14,2009-02-22,PRV52405,5000,PHY369659,PHY392961,PHY349768,2009-02-14,431,1068.0,2009-02-22,067,43491,2762,7843,32723,V1041,4254,25062,40390,4019,NaN,331.0,NaN,NaN,NaN,NaN,NaN,8,9
4,BENE11014,CLM63689,2009-08-13,2009-08-30,PRV56614,10000,PHY379376,PHY398258,NaN,2009-08-13,78321,1068.0,2009-08-30,975,042,3051,34400,5856,42732,486,5119,29620,20300,NaN,3893.0,NaN,NaN,NaN,NaN,NaN,17,9


## Missing Value Analysis - Inpatient Claims

In [31]:
(
    inpatient.isnull()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

ClmProcedureCode_6     40474
ClmProcedureCode_5     40465
ClmProcedureCode_4     40358
ClmProcedureCode_3     39509
ClmDiagnosisCode_10    36547
OtherPhysician         35784
ClmProcedureCode_2     35020
ClmProcedureCode_1     17326
OperatingPhysician     16644
ClmDiagnosisCode_9     13497
ClmDiagnosisCode_8      9942
ClmDiagnosisCode_7      7258
ClmDiagnosisCode_6      4838
ClmDiagnosisCode_5      2894
ClmDiagnosisCode_4      1534
dtype: int64

### Observation

Most missing values occur in diagnosis and procedure code fields.

These missing values are expected because not every claim contains the maximum number of diagnosis or procedure codes. Therefore, they represent the absence of additional diagnoses/procedures rather than data quality issues.

## Missing Value Percentage - Inpatient Claims

In [32]:
(
    inpatient.isnull()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .head(15)
)

ClmProcedureCode_6     100.000000
ClmProcedureCode_5      99.977764
ClmProcedureCode_4      99.713396
ClmProcedureCode_3      97.615753
ClmDiagnosisCode_10     90.297475
OtherPhysician          88.412314
ClmProcedureCode_2      86.524683
ClmProcedureCode_1      42.807728
OperatingPhysician      41.122696
ClmDiagnosisCode_9      33.347334
ClmDiagnosisCode_8      24.563918
ClmDiagnosisCode_7      17.932500
ClmDiagnosisCode_6      11.953353
ClmDiagnosisCode_5       7.150269
ClmDiagnosisCode_4       3.790087
dtype: float64

### Observation

Several procedure code columns contain more than 95% missing values, indicating that most inpatient claims involve only a small number of recorded procedures.

Rather than using individual procedure code fields, feature engineering will focus on aggregate measures such as the number of diagnosis codes and procedure codes per claim.

## Feature Engineering - Length of Stay

In [33]:
inpatient["AdmissionDt"] = pd.to_datetime(inpatient["AdmissionDt"])
inpatient["DischargeDt"] = pd.to_datetime(inpatient["DischargeDt"])
inpatient["LengthOfStay"] = (inpatient["DischargeDt"] - inpatient["AdmissionDt"]).dt.days
inpatient["LengthOfStay"].describe()

count    40474.000000
mean         5.665168
std          5.638538
min          0.000000
25%          2.000000
50%          4.000000
75%          7.000000
max         35.000000
Name: LengthOfStay, dtype: float64

### Observation

The average inpatient length of stay is approximately 5.7 days, with a median stay of 4 days.

Most hospitalizations are relatively short, while a small number of claims involve extended stays of up to 35 days. Length of stay is expected to be an important indicator of healthcare utilization and claim reimbursement levels.

## Length of Stay Distribution

In [34]:
fig = px.histogram(
    inpatient,
    x="LengthOfStay",
    nbins=35,
    title="Distribution of Inpatient Length of Stay"
)
fig.update_layout(title_x=0.5)
fig.show()
fig = px.box(
    inpatient,
    y="LengthOfStay",
    title="Length of Stay Outlier Analysis"
)
fig.update_layout(title_x=0.5)
fig.show()

### Observation

The length of stay distribution is right-skewed, with most inpatient admissions lasting between 2 and 7 days and a median stay of 4 days.

The box plot identifies a small number of extended hospitalizations exceeding 14 days, with the maximum stay reaching 35 days. These longer stays likely represent more complex medical cases and may contribute to higher reimbursement amounts.

Overall, the feature appears realistic and suitable for provider-level healthcare utilization analysis.

## Feature Engineering - Diagnosis Count

In [35]:
diagnosis_cols = [
    "ClmDiagnosisCode_1",
    "ClmDiagnosisCode_2",
    "ClmDiagnosisCode_3",
    "ClmDiagnosisCode_4",
    "ClmDiagnosisCode_5",
    "ClmDiagnosisCode_6",
    "ClmDiagnosisCode_7",
    "ClmDiagnosisCode_8",
    "ClmDiagnosisCode_9",
    "ClmDiagnosisCode_10"
]
inpatient["DiagnosisCount"] = (inpatient[diagnosis_cols].notna().sum(axis=1))
inpatient["DiagnosisCount"].describe()

count    40474.000000
mean         8.087365
std          1.851830
min          1.000000
25%          8.000000
50%          9.000000
75%          9.000000
max         10.000000
Name: DiagnosisCount, dtype: float64

### Observation

Inpatient claims contain a high number of diagnosis codes, with a median of 9 diagnoses per claim.

This indicates that hospitalized beneficiaries often present with multiple comorbidities and complex medical conditions. Diagnosis count serves as a useful proxy for claim complexity and may help distinguish normal provider behavior from potentially fraudulent billing patterns.

## Diagnosis Count Distribution

In [36]:
diag_dist = (
    inpatient["DiagnosisCount"]
    .value_counts()
    .sort_index()
    .reset_index()
)
diag_dist.columns = ["DiagnosisCount", "Count"]
fig = px.bar(
    diag_dist,
    x="DiagnosisCount",
    y="Count",
    text="Count",
    title="Distribution of Diagnosis Count"
)
fig.update_layout(title_x=0.5)
fig.show()

### Observation

Diagnosis count is heavily concentrated between 8 and 10 diagnoses per inpatient claim, with the highest frequency occurring at 9 diagnoses.

This indicates that inpatient claims typically involve medically complex beneficiaries with multiple documented conditions. Diagnosis count can serve as a useful proxy for claim complexity and healthcare resource utilization.

## Summary

Data understanding and claim-level exploratory analysis completed.

Key claim-level features identified:
- Age
- Total Chronic Conditions
- IsDeceased
- Length Of Stay
- Diagnosis Count

These features will be aggregated at the provider level in the next notebook.